# 09_v5c_hedge_tilt — V5C 3.1 vs 3.1a (对冲层内部调整)

> 测试假设: 从 GLDM 拿 5% 给 BCX, 对冲层从 (黄金 20% / 商品 10%) → (黄金 15% / 商品 15%)

**结构变化**:
- V5C 3.1: GLDM 20% / BCX 10% (纯黄金主导的对冲)
- V5C 3.1a: GLDM 15% / BCX 15% (黄金+商品平衡的对冲)

**其他不变**: 进攻 50% / 对冲 30% / 防御 20%

## 关键测试问题

1. **2008 GFC** 怎么变？
   - GLDM 2008: 约 +5% (避险)
   - BCX 类商品 2008: 约 -55% (随股市暴跌)
   - 减黄金加商品 → 大概率恶化 Max DD

2. **2020-2022 通胀期** 怎么变？
   - 商品在通胀环境表现优异
   - 这段时间 V5C 3.1a 应当更好

3. **整体 CAGR / Sharpe** 谁赢？
   - 取决于 23.8 年内 "危机次数 vs 通胀期长度"

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 复用 08 的长史代理和合成逻辑
tickers_long = {
    'VFINX': 'VFINX', 'QQQ': 'QQQ', 'HQH': 'HQH', 'XLV': 'XLV',
    'VFITX': 'VFITX', 'GLD': 'GLD', 'GC=F': 'GC=F',
    'DBC': 'DBC', 'PCRIX': 'PCRIX',
}

raw = yf.download(list(tickers_long.values()), start='1999-01-01', auto_adjust=True)['Close']

# 合成长史数据
def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

gold_long = synthesize(raw['GLD'], raw['GC=F'])
commod_long = synthesize(raw['DBC'], raw['PCRIX'])

long_data = pd.DataFrame({
    'VOO': raw['VFINX'], 'QQQ': raw['QQQ'],
    'HQH': raw['HQH'], 'XLV': raw['XLV'],
    'VGSH': raw['VFITX'], 'GLDM': gold_long, 'BCX': commod_long,
}).dropna()
returns = long_data.pct_change().dropna()
print(f'数据: {long_data.index[0].date()} → {long_data.index[-1].date()} ({len(long_data)/252:.1f} 年)')

In [ ]:
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub_returns = returns_df[used]
    target = np.array([target_weights[t] for t in used])
    target = target / target.sum()
    current_weights = target.copy()
    portfolio_returns = []
    rebalance_dates = [sub_returns.index[0]]
    for date, daily_ret in sub_returns.iterrows():
        port_ret = np.sum(current_weights * daily_ret.values)
        portfolio_returns.append(port_ret)
        new_weights = current_weights * (1 + daily_ret.values)
        new_weights = new_weights / new_weights.sum()
        if np.max(np.abs(new_weights - target)) * 100 >= threshold_pp:
            current_weights = target.copy()
            rebalance_dates.append(date)
        else:
            current_weights = new_weights
    return pd.Series(portfolio_returns, index=sub_returns.index), rebalance_dates

def compute_metrics(returns_series, rebal_dates, name):
    cum = (1 + returns_series).cumprod()
    n_years = len(returns_series) / 252
    cagr = cum.iloc[-1] ** (1/n_years) - 1
    vol = returns_series.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol
    downside = returns_series[returns_series < 0]
    sortino = (cagr - 0.04) / (downside.std() * np.sqrt(252))
    rolling_max = cum.expanding().max()
    drawdown = (cum / rolling_max) - 1
    max_dd = drawdown.min()
    max_dd_date = drawdown.idxmin()
    return {
        'Name': name, 'CAGR': cagr, 'Vol': vol,
        'Sharpe': sharpe, 'Sortino': sortino,
        'Max DD': max_dd, 'Max DD Date': max_dd_date,
        'Calmar': cagr / abs(max_dd),
        'Rebalances': len(rebal_dates) - 1,
    }

In [ ]:
# ============================================================
# 两个组合对比
# ============================================================
V5C_3_1 = {
    'VOO': 0.15, 'QQQ': 0.15, 'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.20, 'BCX': 0.10, 'VGSH': 0.20,
}

V5C_3_1a = {
    'VOO': 0.15, 'QQQ': 0.15, 'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.15, 'BCX': 0.15, 'VGSH': 0.20,  # 改: 黄金 -5pp, 商品 +5pp
}

ret_31, dates_31 = simulate_rebalance(returns, V5C_3_1, threshold_pp=5.0)
ret_31a, dates_31a = simulate_rebalance(returns, V5C_3_1a, threshold_pp=5.0)

m31 = compute_metrics(ret_31, dates_31, 'V5C 3.1 (GLDM 20 / BCX 10)')
m31a = compute_metrics(ret_31a, dates_31a, 'V5C 3.1a (GLDM 15 / BCX 15)')

print('=' * 78)
print('指标对比 (23.8 年长史回测, ±5pp 阈值再平衡)')
print('=' * 78)
for col in ['CAGR', 'Vol', 'Sharpe', 'Sortino', 'Max DD', 'Calmar']:
    fmt = '{:.2%}' if col not in ['Sharpe','Sortino','Calmar'] else '{:.3f}'
    a = m31[col]; b = m31a[col]
    arrow = '↑' if b > a else ('↓' if b < a else '=')
    delta = b - a
    delta_str = fmt.format(delta) if isinstance(delta, float) else str(delta)
    print(f'  {col:<10}  3.1: {fmt.format(a):>10}   3.1a: {fmt.format(b):>10}   Δ: {delta_str:>10}  {arrow}')

print(f"\n  Rebalances: 3.1: {m31['Rebalances']}  3.1a: {m31a['Rebalances']}")
print(f"  Max DD 日期: 3.1: {m31['Max DD Date'].date()}  3.1a: {m31a['Max DD Date'].date()}")

In [ ]:
# ============================================================
# 净值 + 回撤可视化
# ============================================================
cum_31 = (1 + ret_31).cumprod()
cum_31a = (1 + ret_31a).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].plot(cum_31, label='V5C 3.1 (GLDM 20 / BCX 10)', linewidth=2, alpha=0.85)
axes[0].plot(cum_31a, label='V5C 3.1a (GLDM 15 / BCX 15)', linewidth=2, alpha=0.85)
axes[0].set_title('净值对比 (log scale)', fontsize=14)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for cum, label, color in [(cum_31, '3.1', 'C0'), (cum_31a, '3.1a', 'C1')]:
    rm = cum.expanding().max()
    dd = (cum / rm) - 1
    axes[1].fill_between(dd.index, dd.values, 0, alpha=0.4, color=color, label=label)
axes[1].set_title('回撤对比', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 关键时期表现 (重点看 2008 + 通胀期)
# ============================================================
events = {
    '2008 GFC':                ('2007-10-09', '2009-03-09'),
    '2008 GFC 急跌':            ('2008-09-01', '2008-12-31'),
    '2010-2013 商品超级周期':    ('2010-01-01', '2013-12-31'),
    '2014-2019 商品熊市':       ('2014-01-01', '2019-12-31'),
    '2020 COVID':              ('2020-02-19', '2020-04-30'),
    '2020-2021 通胀+反弹':      ('2020-04-30', '2021-12-31'),
    '2022 通胀+加息':           ('2022-01-01', '2022-12-31'),
    '2023-2024 复苏':          ('2023-01-01', '2024-12-31'),
    '2025-2026 高位':          ('2025-01-01', '2026-04-30'),
}

print('=' * 80)
print('关键时期表现 (3.1a vs 3.1)')
print('=' * 80)
print(f'{"时期":<26} {"3.1":>10} {"3.1a":>10} {"Δ (3.1a-3.1)":>14}  解读')
print('-' * 80)
for name, (s, e) in events.items():
    if pd.Timestamp(s) < ret_31.index[0]:
        continue
    a = (1 + ret_31.loc[s:e]).prod() - 1
    b = (1 + ret_31a.loc[s:e]).prod() - 1
    delta = b - a
    interp = '商品赢' if delta > 0.005 else ('黄金赢' if delta < -0.005 else '中性')
    print(f'{name:<26} {a:>+9.2%}  {b:>+9.2%}    {delta:>+9.2%}    {interp}')

In [ ]:
# ============================================================
# GLDM vs BCX 单标的相关性 + 历史 CAGR
# ============================================================
gldm_ret = returns['GLDM']
bcx_ret = returns['BCX']
voo_ret = returns['VOO']

print('=' * 60)
print('GLDM vs BCX 单标的特征 (23.8 年)')
print('=' * 60)
for name, r in [('GLDM (黄金)', gldm_ret), ('BCX (商品)', bcx_ret)]:
    cum = (1 + r).cumprod()
    cagr = cum.iloc[-1] ** (252/len(r)) - 1
    vol = r.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    corr_voo = r.corr(voo_ret)
    corr_gldm = r.corr(gldm_ret)
    print(f'\n{name}:')
    print(f'  CAGR:        {cagr:.2%}')
    print(f'  Vol:         {vol:.2%}')
    print(f'  Sharpe:      {sharpe:.3f}')
    print(f'  Max DD:      {dd:.2%}')
    print(f'  vs VOO 相关性: {corr_voo:.3f}')

corr_gb = gldm_ret.corr(bcx_ret)
print(f'\nGLDM vs BCX 相关性: {corr_gb:.3f}')
print(f'(低相关性 → 两者在不同环境提供保护; 高相关性 → 重叠)')

## 决策标准

**采纳 V5C 3.1a (黄金减半 → 商品加半)** 仅当:
- 整体 Sharpe ≥ V5C 3.1 (即风险调整后回报不变差)
- 2008 GFC 期间 Max DD 恶化 < 5pp (商品在 2008 暴跌的代价可控)
- 2022 通胀期表现 显著 优于 3.1

**保持 V5C 3.1** 如果:
- Sharpe 下降 > 0.05
- 2008 Max DD 加深 > 5pp (说明牺牲了关键的危机保护)
- 通胀期增益 不足以补偿 危机期损失